# ***Packages***

In [ ]:
%pip install torch-geometric
%pip install scanpy
%pip install cell-gears

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.5/144.5 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 119.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.1/53.1 kB 5.3 MB/s eta 0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 111.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 101.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# ***Imports and Loading Dataset***

In [ ]:
import torch
from gears import PertData, GEARS
import scanpy as sc
import sys

# read the data
sys.path.append('../')
adata = sc.read("data/GSE90546/perturb_processed.h5ad")
print(adata)

AnnData object with n_obs × n_vars = 68603 × 5060
    obs: 'condition', 'cell_type', 'dose_val', 'control', 'condition_name'
    var: 'gene_name'
    uns: 'non_dropout_gene_idx', 'non_zeros_gene_idx', 'rank_genes_groups_cov_all', 'top_non_dropout_de_20', 'top_non_zero_de_20'


# ***Normalization & Subsetting 500 most variable genes***

In [ ]:
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata,n_top_genes=500, subset=True)

# ***Creating Dataloader***

In [ ]:
pert_data = PertData('./data')
pert_data.new_data_process(dataset_name = 'GSE90546', adata = adata)
pert_data.load(data_path = './data/GSE90546')
pert_data.prepare_split(split = 'simulation', seed = 1)
pert_data.get_dataloader(batch_size = 32, test_batch_size = 128)

# the testing, training, and validation split
print(pert_data.adata.obs["split"].value_counts())

Downloading...
100%|██████████| 9.46M/9.46M [00:00<00:00, 34.3MiB/s]
Downloading...
100%|██████████| 559k/559k [00:00<00:00, 5.21MiB/s]
Creating pyg object for each cell in the data...
Creating dataset file...
  9%|▉         | 8/87 [00:09<01:34,  1.20s/it]

SRPR+ctrl


 14%|█▍        | 12/87 [00:15<01:40,  1.34s/it]

SLMO2+ctrl


 16%|█▌        | 14/87 [00:17<01:29,  1.22s/it]

...

AMIGO3+ctrl


 67%|██████▋   | 58/87 [01:01<00:29,  1.00s/it]

KCTD16+ctrl


100%|██████████| 87/87 [01:26<00:00,  1.01it/s]
Done!
Saving new dataset pyg object at ./data/gse90546/data_pyg/cell_graphs.pkl
Done!
Found local copy...
These perturbations are not in the GO graph and their perturbation can thus not be predicted
['SRPR+ctrl' 'SLMO2+ctrl' 'TIMM23+ctrl' 'AMIGO3+ctrl' 'KCTD16+ctrl']
Creating pyg object for each cell in the data...
Creating dataset file...
100%|██████████| 82/82 [01:11<00:00,  1.15it/s]
Done!
Saving new dataset pyg object at ./data/GSE90546/data_pyg/cell_graphs.pkl
Done!
Creating new splits....
Saving new splits at ./data/GSE90546/splits/GSE90546_simulation_1_0.75.pkl
Simulation split test composition:
combo_seen0:0
combo_seen1:0
combo_seen2:0
unseen_single:21
Done!
Creating dataloaders....
Done!


split
train    52730
test      9741
val       3428
Name: count, dtype: int64


# ***Experiment One num_similar_genes_go_graph = 10***

In [ ]:
# preparing the model for training
gears_model = GEARS(pert_data, device = 'cuda:0',
                        weight_bias_track = False,
                        proj_name = 'pertnet',
                        exp_name = 'pertnet')
gears_model.model_initialize(hidden_size = 64,
                             num_similar_genes_go_graph = 10)

Found local copy...


In [ ]:
# train the model
gears_model.train(epochs = 7, lr = 1e-3)

Start Training...
Epoch 1 Step 1 Train Loss: 0.5302
Epoch 1 Step 51 Train Loss: 0.5588
Epoch 1 Step 101 Train Loss: 0.5909
Epoch 1 Step 151 Train Loss: 0.6249
Epoch 1 Step 201 Train Loss: 0.6035
Epoch 1 Step 251 Train Loss: 0.5935
Epoch 1 Step 301 Train Loss: 0.4921
Epoch 1 Step 351 Train Loss: 0.5688
Epoch 1 Step 401 Train Loss: 0.5599
Epoch 1 Step 451 Train Loss: 0.5251
Epoch 1 Step 501 Train Loss: 0.5254
Epoch 1 Step 551 Train Loss: 0.6159
Epoch 1 Step 601 Train Loss: 0.5725
Epoch 1 Step 651 Train Loss: 0.4717
Epoch 1 Step 701 Train Loss: 0.5888
Epoch 1 Step 751 Train Loss: 0.5976
Epoch 1 Step 801 Train Loss: 0.5256
Epoch 1 Step 851 Train Loss: 0.4924
Epoch 1 Step 901 Train Loss: 0.5815
Epoch 1 Step 951 Train Loss: 0.4738
Epoch 1 Step 1001 Train Loss: 0.5657
Epoch 1 Step 1051 Train Loss: 0.6190
Epoch 1 Step 1101 Train Loss: 0.5019
Epoch 1 Step 1151 Train Loss: 0.5471
Epoch 1 Step 1201 Train Loss: 0.4407
Epoch 1 Step 1251 Train Loss: 0.5187
Epoch 1 Step 1301 Train Loss: 0.4890
Epoch 

# ***Experiment Two num_similar_genes_go_graph = 30***

In [ ]:
# preparing the model for training
gears_model = GEARS(pert_data, device = 'cuda:0',
                        weight_bias_track = False,
                        proj_name = 'pertnet',
                        exp_name = 'pertnet')
gears_model.model_initialize(hidden_size = 64,
                             num_similar_genes_go_graph = 30)

Found local copy...


In [ ]:
# train the model
gears_model.train(epochs = 7, lr = 1e-3)

Start Training...
Epoch 1 Step 1 Train Loss: 0.5183
Epoch 1 Step 51 Train Loss: 0.5062
Epoch 1 Step 101 Train Loss: 0.5673
Epoch 1 Step 151 Train Loss: 0.5802
Epoch 1 Step 201 Train Loss: 0.6305
Epoch 1 Step 251 Train Loss: 0.6214
Epoch 1 Step 301 Train Loss: 0.4920
Epoch 1 Step 351 Train Loss: 0.5494
Epoch 1 Step 401 Train Loss: 0.4153
Epoch 1 Step 451 Train Loss: 0.5140
Epoch 1 Step 501 Train Loss: 0.5237
Epoch 1 Step 551 Train Loss: 0.5895
Epoch 1 Step 601 Train Loss: 0.6522
Epoch 1 Step 651 Train Loss: 0.5275
Epoch 1 Step 701 Train Loss: 0.5833
Epoch 1 Step 751 Train Loss: 0.6230
Epoch 1 Step 801 Train Loss: 0.5343
Epoch 1 Step 851 Train Loss: 0.5005
Epoch 1 Step 901 Train Loss: 0.6399
Epoch 1 Step 951 Train Loss: 0.5213
Epoch 1 Step 1001 Train Loss: 0.6256
Epoch 1 Step 1051 Train Loss: 0.5584
Epoch 1 Step 1101 Train Loss: 0.5806
Epoch 1 Step 1151 Train Loss: 0.5550
Epoch 1 Step 1201 Train Loss: 0.5275
Epoch 1 Step 1251 Train Loss: 0.5101
Epoch 1 Step 1301 Train Loss: 0.5456
Epoch 

# ***Experiment Three num_similar_genes_co_express_graph = 10***

In [ ]:
# preparing the model for training
gears_model = GEARS(pert_data, device = 'cuda:0',
                        weight_bias_track = False,
                        proj_name = 'pertnet',
                        exp_name = 'pertnet')
gears_model.model_initialize(hidden_size = 64,
                             num_similar_genes_co_express_graph = 10)

Found local copy...


In [ ]:
# train the model
gears_model.train(epochs = 7, lr = 1e-3)

Start Training...
Epoch 1 Step 1 Train Loss: 0.5376
Epoch 1 Step 51 Train Loss: 0.6800
Epoch 1 Step 101 Train Loss: 0.4944
Epoch 1 Step 151 Train Loss: 0.6027
Epoch 1 Step 201 Train Loss: 0.5115
Epoch 1 Step 251 Train Loss: 0.5441
Epoch 1 Step 301 Train Loss: 0.5635
Epoch 1 Step 351 Train Loss: 0.6885
Epoch 1 Step 401 Train Loss: 0.5331
Epoch 1 Step 451 Train Loss: 0.5651
Epoch 1 Step 501 Train Loss: 0.5367
Epoch 1 Step 551 Train Loss: 0.5290
Epoch 1 Step 601 Train Loss: 0.5693
Epoch 1 Step 651 Train Loss: 0.5056
Epoch 1 Step 701 Train Loss: 0.5562
Epoch 1 Step 751 Train Loss: 0.5763
Epoch 1 Step 801 Train Loss: 0.5112
Epoch 1 Step 851 Train Loss: 0.4832
Epoch 1 Step 901 Train Loss: 0.4801
Epoch 1 Step 951 Train Loss: 0.5337
Epoch 1 Step 1001 Train Loss: 0.5919
Epoch 1 Step 1051 Train Loss: 0.4908
Epoch 1 Step 1101 Train Loss: 0.5539
Epoch 1 Step 1151 Train Loss: 0.5497
Epoch 1 Step 1201 Train Loss: 0.5593
Epoch 1 Step 1251 Train Loss: 0.5035
Epoch 1 Step 1301 Train Loss: 0.4954
Epoch 

# ***Experiment Four num_similar_genes_co_express_graph = 30***

In [ ]:
# preparing the model for training
gears_model = GEARS(pert_data, device = 'cuda:0',
                        weight_bias_track = False,
                        proj_name = 'pertnet',
                        exp_name = 'pertnet')
gears_model.model_initialize(hidden_size = 64,
                             num_similar_genes_co_express_graph = 30)

Found local copy...


In [ ]:
# train the model
gears_model.train(epochs = 7, lr = 1e-3)

Start Training...
Epoch 1 Step 1 Train Loss: 0.5301
Epoch 1 Step 51 Train Loss: 0.5279
Epoch 1 Step 101 Train Loss: 0.4831
Epoch 1 Step 151 Train Loss: 0.4624
Epoch 1 Step 201 Train Loss: 0.5533
Epoch 1 Step 251 Train Loss: 0.5350
Epoch 1 Step 301 Train Loss: 0.5370
Epoch 1 Step 351 Train Loss: 0.5361
Epoch 1 Step 401 Train Loss: 0.5526
Epoch 1 Step 451 Train Loss: 0.5794
Epoch 1 Step 501 Train Loss: 0.4903
Epoch 1 Step 551 Train Loss: 0.5065
Epoch 1 Step 601 Train Loss: 0.4898
Epoch 1 Step 651 Train Loss: 0.5428
Epoch 1 Step 701 Train Loss: 0.5349
Epoch 1 Step 751 Train Loss: 0.5508
Epoch 1 Step 801 Train Loss: 0.5924
Epoch 1 Step 851 Train Loss: 0.5796
Epoch 1 Step 901 Train Loss: 0.5627
Epoch 1 Step 951 Train Loss: 0.5149
Epoch 1 Step 1001 Train Loss: 0.5747
Epoch 1 Step 1051 Train Loss: 0.5374
Epoch 1 Step 1101 Train Loss: 0.5469
Epoch 1 Step 1151 Train Loss: 0.5530
Epoch 1 Step 1201 Train Loss: 0.5664
Epoch 1 Step 1251 Train Loss: 0.5132
Epoch 1 Step 1301 Train Loss: 0.4963
Epoch 

# ***Additional Runs***

In [ ]:
gears_model = GEARS(pert_data, device = 'cuda:0',
                        weight_bias_track = False,
                        proj_name = 'pertnet',
                        exp_name = 'pertnet')
gears_model.model_initialize(hidden_size = 64,
                             num_similar_genes_go_graph = 10,
                             num_similar_genes_co_express_graph = 10)

Downloading...
100%|██████████| 60.7M/60.7M [00:03<00:00, 18.7MiB/s]
Extracting tar file...
Done!


In [ ]:
gears_model.train(epochs = 10, lr = 1e-3)

Start Training...
Epoch 1 Step 1 Train Loss: 0.5662
Epoch 1 Step 51 Train Loss: 0.5062
Epoch 1 Step 101 Train Loss: 0.5726
Epoch 1 Step 151 Train Loss: 0.5379
Epoch 1 Step 201 Train Loss: 0.5614
Epoch 1 Step 251 Train Loss: 0.5736
Epoch 1 Step 301 Train Loss: 0.6424
Epoch 1 Step 351 Train Loss: 0.6255
Epoch 1 Step 401 Train Loss: 0.4979
Epoch 1 Step 451 Train Loss: 0.6012
Epoch 1 Step 501 Train Loss: 0.5410
Epoch 1 Step 551 Train Loss: 0.7068
Epoch 1 Step 601 Train Loss: 0.5428
Epoch 1 Step 651 Train Loss: 0.5557
Epoch 1 Step 701 Train Loss: 0.5672
Epoch 1 Step 751 Train Loss: 0.4936
Epoch 1 Step 801 Train Loss: 0.5454
Epoch 1 Step 851 Train Loss: 0.5726
Epoch 1 Step 901 Train Loss: 0.5495
Epoch 1 Step 951 Train Loss: 0.5507
Epoch 1 Step 1001 Train Loss: 0.5979
Epoch 1 Step 1051 Train Loss: 0.4744
Epoch 1 Step 1101 Train Loss: 0.5839
Epoch 1 Step 1151 Train Loss: 0.5771
Epoch 1 Step 1201 Train Loss: 0.4706
Epoch 1 Step 1251 Train Loss: 0.5402
Epoch 1 Step 1301 Train Loss: 0.5395
Epoch 